# 🐉 Imagen → 3D con Hunyuan3D-2 (gratis) — con todos los parches

Convierte **una imagen de personaje (SIN FONDO)** en un **modelo 3D** (`.glb`) en la GPU gratis de Colab.
Trae cada parche real que hizo falta para que funcione:

| Problema real | Parche |
|---|---|
| `AttributeError … _blas_supports_fpe` al importar | **Celda 1B**: `numpy>=2.1` + **Reiniciar sesión** |
| Warnings `numba / cuml / cudf incompatible` | inofensivos, se ignoran |
| El widget de subir archivos no anda (sobre todo en **celular**) | **Celda 2**: sube por widget **o** por el panel de Archivos 📁 |
| `Model path not exists` con el **mini** | **Celda 3**: modelo **completo** `tencent/Hunyuan3D-2` |
| `CUDA out of memory` | **Celda 3B**: `octree_resolution=192` |

> ⚠️ **Regla de oro:** subí la imagen **YA SIN FONDO** (PNG transparente, recortada). Con fondo, el modelo lo interpreta como cuerpo y sale deforme.

## Orden
1. **GPU T4**: `Entorno de ejecución` → `Cambiar tipo de entorno` → **T4 GPU** → Guardar.
2. **Celda 1** (instalar) → **Celda 1B** (numpy) → **Reiniciar sesión**.
3. **Celda 2** (subir la imagen sin fondo).
4. **Celda 3** (generar). Si da *out of memory* → **Celda 3B**.
5. **Celda 4** (descargar `.glb`).

## Celda 1 — Instalar Hunyuan3D-2
Tarda ~5–8 min la primera vez. Solo lo necesario para la **forma** (no compila los módulos CUDA de textura, que fallan seguido en Colab).

In [ ]:
!nvidia-smi -L

import os
os.chdir('/content')
if not os.path.isdir('/content/Hunyuan3D-2'):
    !git clone https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git
os.chdir('/content/Hunyuan3D-2')

# Dependencias de la generacion de FORMA
!pip install -q ninja
!pip install -q diffusers transformers accelerate trimesh omegaconf einops opencv-python-headless huggingface_hub
!pip install -q -e . 2>&1 | tail -3          # instala hy3dgen (ignora warnings de version)

import torch
print('\ntorch:', torch.__version__, '| GPU:', torch.cuda.is_available())
print('✅ Celda 1 lista. Ahora corré la Celda 1B.')

## Celda 1B — Parche `numpy` → **REINICIAR SESIÓN**
Sin esto, la Celda 3 tira `AttributeError: _blas_supports_fpe` al importar `trimesh`/`hy3dgen` (a `scipy` le falta ese símbolo en numpy < 2.1).
Los warnings de `numba / cuml / cudf incompatible` que aparezcan son **inofensivos** (son de RAPIDS, no los usamos).

In [ ]:
!pip install -q -U "numpy>=2.1"
print('\n⚠️  AHORA hacé:  Entorno de ejecución → Reiniciar sesión')
print('   (obligatorio para que tome el numpy nuevo). Después seguí con la Celda 2.')

## Celda 2 — Subir tu imagen (SIN FONDO)
Dos formas (usá la que te funcione, **en celular la opción B es más confiable**):
- **A)** Corré esta celda y usá el botón *Elegir archivos*.
- **B)** Abrí el panel **Archivos** 📁 (ícono de carpeta a la izquierda), arrastrá/subí tu PNG, y corré esta celda: detecta sola la imagen más reciente en `/content`.

In [ ]:
import os, glob
from PIL import Image

IMG = None
# Opcion A: widget de subida (si anda). files.upload() guarda el archivo en la carpeta actual.
try:
    from google.colab import files
    up = files.upload()
    if up:
        IMG = os.path.abspath(list(up.keys())[0])
except Exception as e:
    print('El widget de subida no anduvo (', e, ') -> uso la opción B.')

# Opcion B: buscar la imagen mas reciente subida por el panel Archivos 📁 (queda en /content)
if not IMG or not os.path.exists(IMG):
    cand = []
    for ext in ('png','jpg','jpeg','webp'):
        cand += glob.glob('/content/*.'+ext)
    cand.sort(key=os.path.getmtime)
    IMG = cand[-1] if cand else None

assert IMG and os.path.exists(IMG), 'No encontré imagen. Subila por el botón de arriba o por el panel Archivos 📁 y volvé a correr esta celda.'
im = Image.open(IMG)
transp = (im.mode == 'RGBA')
print('Imagen:', IMG, '| modo:', im.mode, '| tamaño:', im.size)
print('✅ Transparente (sin fondo).' if transp else '⚠️ NO es transparente. Para que no salga deforme, subí un PNG recortado sin fondo.')

## Celda 3 — Generar el modelo 3D (forma)
Modelo **completo** `tencent/Hunyuan3D-2` (el `mini` da `Model path not exists`).
La **primera vez** baja ~10 GB de pesos → paciencia (barras de descarga). Después genera la malla (~30–60 s en la T4). Exporta a `/content/hunyuan_mesh.glb`.

In [ ]:
import os, torch
os.chdir('/content/Hunyuan3D-2')
from PIL import Image
from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline

print('Cargando el modelo (la 1a vez baja ~10 GB)...')
pipe = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained('tencent/Hunyuan3D-2')
print('✅ Modelo cargado.')

img = Image.open(IMG).convert('RGBA')   # conserva la transparencia (imagen sin fondo)
mesh = pipe(image=img, num_inference_steps=30, octree_resolution=256,
            generator=torch.manual_seed(0))[0]

OUT = '/content/hunyuan_mesh.glb'
os.makedirs(os.path.dirname(OUT), exist_ok=True)   # por las dudas, la carpeta destino existe
mesh.export(OUT)
print('\nResultado:', ('✅ ' + OUT + ' — ' + str(round(os.path.getsize(OUT)/1024, 1)) + ' KB')
      if os.path.exists(OUT) else '❌ no se generó, copiame el error de arriba')

## Celda 3B — Solo si la Celda 3 dio `CUDA out of memory`
Baja la resolución a 192 (menos VRAM). Si aun así falla: `Entorno de ejecución → Reiniciar sesión` y corré esta celda directo (Celda 2 primero para tener `IMG`).

In [ ]:
import os, torch
os.chdir('/content/Hunyuan3D-2')
from PIL import Image
from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline

pipe = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained('tencent/Hunyuan3D-2')
img = Image.open(IMG).convert('RGBA')
mesh = pipe(image=img, num_inference_steps=30, octree_resolution=192,
            generator=torch.manual_seed(0))[0]
OUT = '/content/hunyuan_mesh.glb'
mesh.export(OUT)
print('✅', OUT, round(os.path.getsize(OUT)/1024,1), 'KB' if os.path.exists(OUT) else 'no se generó')

## Celda 4 — Descargar el `.glb`

In [ ]:
from google.colab import files
files.download('/content/hunyuan_mesh.glb')

---
### Resumen de parches
- **`AttributeError _blas_supports_fpe`** → Celda 1B (`numpy>=2.1`) + **Reiniciar sesión**. El error más común: pasa si te saltás el reinicio.
- **Celda 2 no sube nada / celular** → subí por el panel **Archivos** 📁 y corré la Celda 2 (detecta la imagen sola).
- **`Model path not exists` / baja 0 files** → estás con el `mini`. Usá `tencent/Hunyuan3D-2` (Celda 3).
- **Sale deforme / con fondo pegado** → subiste la imagen con fondo. Recortala (PNG transparente) y re-subí.
- **`CUDA out of memory`** → Celda 3B (octree 192) o reiniciar sesión.
- Esto genera **solo la forma** (malla gris, sin color). Para color/textura es un paso extra pesado (avisá y lo armamos).
- Cuando tengas el `.glb`, pasámelo y le pongo las animaciones mocap. Cualquier error rojo, copiámelo. 🐉